# **Cell 1 — Setup: Mount Drive, Seed, Define Paths**

In [1]:
!pip install -q transformers accelerate bitsandbytes peft

from google.colab import drive
drive.mount('/content/drive')

from transformers import set_seed
set_seed(42)

import os
PROJECT_ROOT = '/content/drive/MyDrive/Bengali Paraphrasing'
PROCESSED_DIR = f'{PROJECT_ROOT}/processed_data'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs/generations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

INSTRUCTION = "নিচের বাক্যটি ভিন্নভাবে, একই অর্থ বজায় রেখে লিখুন।"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 12.8 MB/s eta 0:00:00
Mounted at /content/drive


# **Cell 2 — Load the Full, Untouched Test Set**

In [2]:
import pandas as pd

test_df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'test_clean.parquet'))
print(f"Full test set size: {len(test_df)}")

Full test set size: 23332


# **Cell 3 — Load TigerLLM (best_model)**

In [3]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TIGER_BASE_CACHE = f'{PROJECT_ROOT}/base_model_cache/tigerllm'
TIGER_BEST_MODEL = f'{PROJECT_ROOT}/checkpoints/tigerllm/best_model'

tiger_tokenizer = AutoTokenizer.from_pretrained(TIGER_BEST_MODEL)
if tiger_tokenizer.pad_token is None:
    tiger_tokenizer.pad_token = tiger_tokenizer.eos_token
tiger_tokenizer.padding_side = "left"  # mandatory for decoder-only batched generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tiger_base = AutoModelForCausalLM.from_pretrained(TIGER_BASE_CACHE, quantization_config=bnb_config, device_map={"": 0})
tiger_model = PeftModel.from_pretrained(tiger_base, TIGER_BEST_MODEL)
tiger_model.eval()

# Documentation
total_params = sum(p.numel() for p in tiger_model.parameters())
trainable_params = sum(p.numel() for n, p in tiger_model.named_parameters() if "lora_" in n)
print(f"TigerLLM total params: {total_params:,} | trainable (LoRA): {trainable_params:,}")
import transformers

print("transformers_version:", transformers.__version__)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

TigerLLM total params: 664,050,816 | trainable (LoRA): 13,045,760
transformers_version: 5.13.1


# **Cell 4 — Generate with TigerLLM (full test set, batch 128, greedy, resume-safe)**

In [ ]:
TIGER_OUTPUT_PATH = f'{OUTPUT_DIR}/tigerllm_generations.parquet'
batch_size = 128

if os.path.exists(TIGER_OUTPUT_PATH):
    tiger_results_df = pd.read_parquet(TIGER_OUTPUT_PATH)
    done_count = len(tiger_results_df)
    print(f"Resuming: {done_count} examples already generated.")
else:
    tiger_results_df = pd.DataFrame(columns=['source', 'target', 'generated'])
    done_count = 0

remaining_df = test_df.iloc[done_count:].reset_index(drop=True)

new_rows = []
save_every = 10  # batches -- at batch 128, this checkpoints roughly every 1280 examples

for batch_num, i in enumerate(range(0, len(remaining_df), batch_size)):
    batch = remaining_df.iloc[i:i+batch_size]
    prompts = [tiger_tokenizer.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCTION}\n\n{s}"}], tokenize=False, add_generation_prompt=True
    ) for s in batch['source']]
    inputs = tiger_tokenizer(prompts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        output_ids = tiger_model.generate(**inputs, max_new_tokens=150, do_sample=False)

    prompt_len = inputs['input_ids'].shape[1]
    for j, out_ids in enumerate(output_ids):
        generated = tiger_tokenizer.decode(out_ids[prompt_len:], skip_special_tokens=True)
        new_rows.append({
            'source': batch['source'].iloc[j],
            'target': batch['target'].iloc[j],
            'generated': generated
        })

    if (batch_num + 1) % save_every == 0:
        tiger_results_df = pd.concat([tiger_results_df, pd.DataFrame(new_rows)], ignore_index=True)
        tiger_results_df.to_parquet(TIGER_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"Checkpoint saved: {len(tiger_results_df)} / {len(test_df)} generated.")

if new_rows:
    tiger_results_df = pd.concat([tiger_results_df, pd.DataFrame(new_rows)], ignore_index=True)
    tiger_results_df.to_parquet(TIGER_OUTPUT_PATH, index=False)

print(f"TigerLLM generation complete: {len(tiger_results_df)} / {len(test_df)}")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Checkpoint saved: 1280 / 23332 generated.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Checkpoint saved: 2560 / 23332 generated.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

...

Checkpoint saved: 21760 / 23332 generated.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

Checkpoint saved: 23040 / 23332 generated.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TigerLLM generation complete: 23332 / 23332


# Check a sample

In [ ]:
tiger_results_df.sample(5, random_state=42)

,source,target,generated
2864,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...",দীর্ঘ ভ্রমণে ক্লান্ত হয়ে পড়া লোকদের সকালেই ফ...
7638,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,বাড়িতে সাধারণত কোনো ঝগড়া হয় না।
12343,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,"কিছু ভিডিওতে, কল্পিত শত্রুদের প্রতি প্রতিশোধ ন..."
21159,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,"প্রথম দিনে, ২৬ জন লোককে টিকা দেওয়া হয়।"
22798,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,এরপর ডেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়ে।


# **Cell 5 — Load Qwen2.5 (best_model)**

In [4]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QWEN_BASE_CACHE = f'{PROJECT_ROOT}/base_model_cache/qwen2.5'
QWEN_BEST_MODEL = f'{PROJECT_ROOT}/checkpoints/qwen2.5/best_model'

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_BEST_MODEL)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
qwen_tokenizer.padding_side = "left"

qwen_base = AutoModelForCausalLM.from_pretrained(
    QWEN_BASE_CACHE, quantization_config=bnb_config, device_map={"": 0}, attn_implementation="sdpa"
)
qwen_model = PeftModel.from_pretrained(qwen_base, QWEN_BEST_MODEL)
qwen_model.eval()

# Documentation
total_params = sum(p.numel() for p in qwen_model.parameters())
trainable_params = sum(p.numel() for n, p in qwen_model.named_parameters() if "lora_" in n)
print(f"Qwen2.5 total params: {total_params:,} | trainable (LoRA): {trainable_params:,}")
print("transformers_version:", transformers.__version__)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2.5 total params: 907,081,216 | trainable (LoRA): 18,464,768
transformers_version: 5.13.1


# **Cell 6 — Generate with Qwen2.5 (full test set, batch 96, greedy, resume-safe)**

In [5]:
from tqdm import tqdm

QWEN_OUTPUT_PATH = f'{OUTPUT_DIR}/qwen25_generations.parquet'
batch_size = 96

if os.path.exists(QWEN_OUTPUT_PATH):
    qwen_results_df = pd.read_parquet(QWEN_OUTPUT_PATH)
    done_count = len(qwen_results_df)
    print(f"Resuming: {done_count} examples already generated.")
else:
    qwen_results_df = pd.DataFrame(columns=['source', 'target', 'generated'])
    done_count = 0

remaining_df = test_df.iloc[done_count:].reset_index(drop=True)

new_rows = []
save_every = 10  # every 960 examples at batch 96

for batch_num, i in enumerate(tqdm(range(0, len(remaining_df), batch_size), desc="Qwen2.5 generation")):
    batch = remaining_df.iloc[i:i+batch_size]
    prompts = [qwen_tokenizer.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCTION}\n\n{s}"}], tokenize=False, add_generation_prompt=True
    ) for s in batch['source']]
    inputs = qwen_tokenizer(prompts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=150, do_sample=False)

    prompt_len = inputs['input_ids'].shape[1]
    for j, out_ids in enumerate(output_ids):
        generated = qwen_tokenizer.decode(out_ids[prompt_len:], skip_special_tokens=True)
        new_rows.append({
            'source': batch['source'].iloc[j],
            'target': batch['target'].iloc[j],
            'generated': generated
        })

    if (batch_num + 1) % save_every == 0:
        qwen_results_df = pd.concat([qwen_results_df, pd.DataFrame(new_rows)], ignore_index=True)
        qwen_results_df.to_parquet(QWEN_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"Checkpoint saved: {len(qwen_results_df)} / {len(test_df)} generated.")

if new_rows:
    qwen_results_df = pd.concat([qwen_results_df, pd.DataFrame(new_rows)], ignore_index=True)
    qwen_results_df.to_parquet(QWEN_OUTPUT_PATH, index=False)

print(f"Qwen2.5 generation complete: {len(qwen_results_df)} / {len(test_df)}")

Resuming: 20160 examples already generated.


Qwen2.5 generation:  29%|██▉       | 10/34 [09:53<23:27, 58.63s/it]

Checkpoint saved: 21120 / 23332 generated.


Qwen2.5 generation:  59%|█████▉    | 20/34 [20:31<14:48, 63.43s/it]

Checkpoint saved: 22080 / 23332 generated.


Qwen2.5 generation:  88%|████████▊ | 30/34 [30:30<03:55, 58.93s/it]

Checkpoint saved: 23040 / 23332 generated.


Qwen2.5 generation: 100%|██████████| 34/34 [34:00<00:00, 60.01s/it]

Qwen2.5 generation complete: 23332 / 23332


In [6]:
qwen_results_df.sample(5, random_state=42)

,source,target,generated
2864,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...","সবাই দীর্ঘ ভ্রমণে ক্লান্ত হয়েছে, সকালে আবার র..."
7638,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,এই ঘরে সাধারণত কোন ঝগড়া হয় না।
12343,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,কিছু ভিডিওতে কল্পনাগত শত্রুর বিরুদ্ধে পরিবর্তন...
21159,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,প্রথম দিনে ২৬ জনকে টিকা প্রদান করা হয়।
22798,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,দেসিয়াস সেই সময়ে শত্রুদলের মধ্যে পড়াশোনা কর...


In [9]:
ib_df = pd.read_parquet(f'{OUTPUT_DIR}/indicbartss_generations.parquet')
bt5_df = pd.read_parquet(f'{OUTPUT_DIR}/banglat5_generations.parquet')
tiger_df = pd.read_parquet(f'{OUTPUT_DIR}/tigerllm_generations.parquet')
qwen_df = pd.read_parquet(f'{OUTPUT_DIR}/qwen25_generations.parquet')

# **CELL 7 - row-count/alignment check**

In [10]:
for name, df in [('IndicBARTSS', ib_df), ('BanglaT5', bt5_df), ('TigerLLM', tiger_df), ('Qwen2.5', qwen_df)]:
    print(f"{name}: {len(df)} rows")

# Confirm source/target columns are identical and in the same order across all four
assert (ib_df['source'].values == bt5_df['source'].values).all(), "IndicBARTSS vs BanglaT5 source mismatch!"
assert (ib_df['source'].values == tiger_df['source'].values).all(), "IndicBARTSS vs TigerLLM source mismatch!"
assert (ib_df['source'].values == qwen_df['source'].values).all(), "IndicBARTSS vs Qwen2.5 source mismatch!"
assert (ib_df['target'].values == bt5_df['target'].values).all(), "IndicBARTSS vs BanglaT5 target mismatch!"
assert (ib_df['target'].values == tiger_df['target'].values).all(), "IndicBARTSS vs TigerLLM target mismatch!"
assert (ib_df['target'].values == qwen_df['target'].values).all(), "IndicBARTSS vs Qwen2.5 target mismatch!"
print("\nAll four files aligned: same row count, same source/target order.")

IndicBARTSS: 23332 rows
BanglaT5: 23332 rows
TigerLLM: 23332 rows
Qwen2.5: 23332 rows

All four files aligned: same row count, same source/target order.


# **CELL 8 - truncation check**

In [11]:
def check_truncation(df, tokenizer, max_new_tokens=150):
    truncated_count = 0
    for gen in df['generated']:
        token_count = len(tokenizer(gen, add_special_tokens=False)['input_ids'])
        if token_count >= max_new_tokens - 2:  # small buffer since exact boundary can vary slightly
            truncated_count += 1
    pct = truncated_count / len(df) * 100
    return truncated_count, pct

tiger_trunc, tiger_pct = check_truncation(tiger_df, tiger_tokenizer)
qwen_trunc, qwen_pct = check_truncation(qwen_df, qwen_tokenizer)
print(f"TigerLLM: {tiger_trunc} / {len(tiger_df)} likely truncated ({tiger_pct:.2f}%)")
print(f"Qwen2.5: {qwen_trunc} / {len(qwen_df)} likely truncated ({qwen_pct:.2f}%)")

TigerLLM: 4 / 23332 likely truncated (0.02%)
Qwen2.5: 1219 / 23332 likely truncated (5.22%)


# **CELL 9 - degenerate output check**

In [12]:
def check_degenerate(df):
    empty_count = (df['generated'].str.strip() == '').sum()
    exact_echo_count = (df['generated'].str.strip() == df['source'].str.strip()).sum()
    # crude repetition check: does any single word repeat 5+ times consecutively?
    import re
    def has_repetition_loop(text, min_repeats=5):
        words = text.split()
        for i in range(len(words) - min_repeats + 1):
            if len(set(words[i:i+min_repeats])) == 1:
                return True
        return False
    repetition_count = df['generated'].apply(has_repetition_loop).sum()
    return empty_count, exact_echo_count, repetition_count

for name, df in [('TigerLLM', tiger_df), ('Qwen2.5', qwen_df)]:
    empty, echo, rep = check_degenerate(df)
    print(f"{name}: {empty} empty | {echo} exact source echoes | {rep} repetition-loop outputs")

TigerLLM: 0 empty | 53 exact source echoes | 3 repetition-loop outputs
Qwen2.5: 0 empty | 6 exact source echoes | 2 repetition-loop outputs


# **generation-config documentation**

In [13]:
decoder_model_documentation = {
    "generation_settings": {
        "decoding_strategy": "greedy (do_sample=False)",
        "max_new_tokens": 150,
        "instruction_prompt": INSTRUCTION,
        "prompt_format": "tokenizer.apply_chat_template with role='user', add_generation_prompt=True",
        "padding_side": "left (mandatory for batched causal LM generation)",
    },
    "TigerLLM": {
        "total_params": sum(p.numel() for p in tiger_model.parameters()),
        "trainable_lora_params": sum(p.numel() for n, p in tiger_model.named_parameters() if "lora_" in n),
        "transformers_version": transformers.__version__,
        "batch_size_used": 128,
        "truncated_generations": f"{tiger_trunc} ({tiger_pct:.2f}%)",
    },
    "Qwen2.5": {
        "total_params": sum(p.numel() for p in qwen_model.parameters()),
        "trainable_lora_params": sum(p.numel() for n, p in qwen_model.named_parameters() if "lora_" in n),
        "transformers_version": transformers.__version__,
        "batch_size_used": 96,
        "truncated_generations": f"{qwen_trunc} ({qwen_pct:.2f}%)",
    },
}

doc_path = f'{OUTPUT_DIR}/decoder_model_documentation.json'
with open(doc_path, 'w', encoding='utf-8') as f:
    json.dump(decoder_model_documentation, f, ensure_ascii=False, indent=2)
print(json.dumps(decoder_model_documentation, ensure_ascii=False, indent=2))

{
  "generation_settings": {
    "decoding_strategy": "greedy (do_sample=False)",
    "max_new_tokens": 150,
    "instruction_prompt": "নিচের বাক্যটি ভিন্নভাবে, একই অর্থ বজায় রেখে লিখুন।",
    "prompt_format": "tokenizer.apply_chat_template with role='user', add_generation_prompt=True",
    "padding_side": "left (mandatory for batched causal LM generation)"
  },
  "TigerLLM": {
    "total_params": 664050816,
    "trainable_lora_params": 13045760,
    "transformers_version": "5.13.1",
    "batch_size_used": 128,
    "truncated_generations": "4 (0.02%)"
  },
  "Qwen2.5": {
    "total_params": 907081216,
    "trainable_lora_params": 18464768,
    "transformers_version": "5.13.1",
    "batch_size_used": 96,
    "truncated_generations": "1219 (5.22%)"
  }
}


In [14]:
def get_flagged_indices(df):
    empty_idx = df[df['generated'].str.strip() == ''].index.tolist()
    echo_idx = df[df['generated'].str.strip() == df['source'].str.strip()].index.tolist()
    def has_repetition_loop(text, min_repeats=5):
        words = text.split()
        for i in range(len(words) - min_repeats + 1):
            if len(set(words[i:i+min_repeats])) == 1:
                return True
        return False
    rep_idx = df[df['generated'].apply(has_repetition_loop)].index.tolist()
    return {"empty": empty_idx, "exact_echo": echo_idx, "repetition_loop": rep_idx}

flagged = {
    "TigerLLM": get_flagged_indices(tiger_df),
    "Qwen2.5": get_flagged_indices(qwen_df),
}

flagged_path = f'{OUTPUT_DIR}/flagged_degenerate_outputs.json'
with open(flagged_path, 'w', encoding='utf-8') as f:
    json.dump(flagged, f, ensure_ascii=False, indent=2)
print("Saved flagged indices to:", flagged_path)

Saved flagged indices to: /content/drive/MyDrive/Bengali Paraphrasing/outputs/generations/flagged_degenerate_outputs.json


# ***Final - The Qualitative Comparison Table***

In [15]:
# Load all four generation files
ib_df = pd.read_parquet(f'{OUTPUT_DIR}/indicbartss_generations.parquet')
bt5_df = pd.read_parquet(f'{OUTPUT_DIR}/banglat5_generations.parquet')
tiger_df = pd.read_parquet(f'{OUTPUT_DIR}/tigerllm_generations.parquet')
qwen_df = pd.read_parquet(f'{OUTPUT_DIR}/qwen25_generations.parquet')

# Use the same 5 fixed indices we've been spot-checking throughout
fixed_indices = [2864, 7638, 12343, 21159, 22798]

comparison_table = pd.DataFrame({
    'Source': ib_df.loc[fixed_indices, 'source'].values,
    'Reference (target)': ib_df.loc[fixed_indices, 'target'].values,
    'IndicBARTSS': ib_df.loc[fixed_indices, 'generated'].values,
    'BanglaT5': bt5_df.loc[fixed_indices, 'generated'].values,
    'TigerLLM': tiger_df.loc[fixed_indices, 'generated'].values,
    'Qwen2.5': qwen_df.loc[fixed_indices, 'generated'].values,
})

comparison_path = f'{OUTPUT_DIR}/qualitative_comparison_table.csv'
comparison_table.to_csv(comparison_path, index=False, encoding='utf-8-sig')
print("Saved to:", comparison_path)
comparison_table

Saved to: /content/drive/MyDrive/Bengali Paraphrasing/outputs/generations/qualitative_comparison_table.csv


,Source,Reference (target),IndicBARTSS,BanglaT5,TigerLLM,Qwen2.5
0,"দীর্ঘ ভ্রমণে ক্লান্ত সবাই, সকালেই রওনা দিতে হব...","দীর্ঘ যাত্রায় ক্লান্ত, সকালে আমাদের চলে যেতে ...","দীর্ঘ ভ্রমণ শেষে, সবাই সকালে আবার ফিরে আসবে।","দীর্ঘ ভ্রমণের কারণে ক্লান্ত সবাই, সকালে রওনা দ...",দীর্ঘ ভ্রমণে ক্লান্ত হয়ে পড়া লোকদের সকালেই ফ...,"সবাই দীর্ঘ ভ্রমণে ক্লান্ত হয়েছে, সকালে আবার র..."
1,এ বাড়িতে সাধারণত কোনো ঝগড়াঝাঁটি হয় না।,এ বাড়িতে কোনো ঝগড়া নেই।,এ বাড়িতে সাধারণত কোনো ঝগড়া হয় না।,এ বাড়িতে সাধারণত কোনো ঝগড়া হয় না।,বাড়িতে সাধারণত কোনো ঝগড়া হয় না।,এই ঘরে সাধারণত কোন ঝগড়া হয় না।
2,কিছু ভিডিওতে কল্পিত শত্রুর বিরুদ্ধে বদলা নেওয়...,কিছু ভিডিও কাল্পনিক শত্রুদের বিরুদ্ধে প্রতিশোধ...,কিছু ভিডিওতে শত্রুর বিরুদ্ধে শত্রুর বিরুদ্ধে য...,কিছু ভিডিওতে কাল্পনিক শত্রুর বিরুদ্ধে প্রতিশোধ...,"কিছু ভিডিওতে, কল্পিত শত্রুদের প্রতি প্রতিশোধ ন...",কিছু ভিডিওতে কল্পনাগত শত্রুর বিরুদ্ধে পরিবর্তন...
3,প্রথম দিনে মোট ২৬ জনকে টিকা দেয়া হয়।,প্রথম দিন সর্বমোট ২৬ জনকে অনাক্রম্য করা হয়।,প্রথম দিনে ২৬ জন টিকা প্রদান করা হয়।,প্রথম দিনে ২৬ জনকে টিকা দেওয়া হয়।,"প্রথম দিনে, ২৬ জন লোককে টিকা দেওয়া হয়।",প্রথম দিনে ২৬ জনকে টিকা প্রদান করা হয়।
4,দেসিয়াস তখন শত্রুদলের মধ্যে ঝাঁপ দেন।,এরপর দেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়েছিলেন।,দেসিয়াস শত্রুদের বিরুদ্ধে যুদ্ধে ঝাঁপিয়ে পড়ে।,দেসিয়াস তখন শত্রু দলের মধ্যে ঝাঁপ দেন।,এরপর ডেসিয়াস শত্রুদের মধ্যে ঝাঁপিয়ে পড়ে।,দেসিয়াস সেই সময়ে শত্রুদলের মধ্যে পড়াশোনা কর...


In [16]:
!pip install -q kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.5 MB/s eta 0:00:00


In [19]:
!pip install -U kaleido

# Qualitative comparison table — Plotly version

In [22]:
import plotly.graph_objects as go

fixed_indices = [2864, 7638, 12343, 21159, 22798]

sources = ib_df.loc[fixed_indices, 'source'].values
targets = ib_df.loc[fixed_indices, 'target'].values
ib_gen = ib_df.loc[fixed_indices, 'generated'].values
bt5_gen = bt5_df.loc[fixed_indices, 'generated'].values
tiger_gen = tiger_df.loc[fixed_indices, 'generated'].values
qwen_gen = qwen_df.loc[fixed_indices, 'generated'].values

fig = go.Figure(data=[go.Table(
    columnwidth=[180, 180, 160, 160, 160, 160],
    header=dict(
        values=['<b>Source</b>', '<b>Reference</b>', '<b>IndicBARTSS</b>', '<b>BanglaT5</b>', '<b>TigerLLM</b>', '<b>Qwen2.5</b>'],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='left',
        height=36
    ),
    cells=dict(
        values=[sources, targets, ib_gen, bt5_gen, tiger_gen, qwen_gen],
        fill_color=[['#f8f9fa', '#e9ecef']*3],
        align='left',
        font=dict(size=12),
        height=60
    )
)])

fig.update_layout(
    title='Qualitative Comparison: All Four Models on Fixed Evaluation Sentences',
    width=1400, height=450,
    margin=dict(l=10, r=10, t=50, b=10)
)

fig.show()

# Save BOTH: interactive HTML (for exploration/presentations) and static SVG/PNG (for the paper)
fig.write_html(f'{OUTPUT_DIR}/qualitative_comparison_table.html')
fig.write_image(f'{OUTPUT_DIR}/qualitative_comparison_table.svg')
fig.write_image(f'{OUTPUT_DIR}/qualitative_comparison_table.png', scale=2)

print("Saved HTML, SVG, and PNG versions.")

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [23]:
import os

output_files = os.listdir(OUTPUT_DIR)
print(f"Files in {OUTPUT_DIR}:")

svg_found = False
png_found = False
html_found = False

for file in output_files:
    print(file)
    if file == 'qualitative_comparison_table.svg':
        svg_found = True
    elif file == 'qualitative_comparison_table.png':
        png_found = True
    elif file == 'qualitative_comparison_table.html':
        html_found = True

if svg_found and png_found and html_found:
    print("\nAll expected output files (HTML, SVG, PNG) were found.")
else:
    print("\nWarning: Some expected output files (HTML, SVG, PNG) were NOT found.")

Files in /content/drive/MyDrive/Bengali Paraphrasing/outputs/generations:
indicbartss_generations.parquet
banglat5_generations.parquet
seq2seq_model_documentation.json
tigerllm_generations.parquet
qwen25_generations.parquet
decoder_model_documentation.json
flagged_degenerate_outputs.json
qualitative_comparison_table.csv
qualitative_comparison_table.html



In [21]:
try:
    import kaleido
    print("'kaleido' package imported successfully.")
except ImportError:
    print("'kaleido' package could not be imported. Please ensure it is installed and the runtime is restarted.")

'kaleido' package imported successfully.
